In [0]:
import dlt
from pyspark.sql.functions import expr

# ==============================================================================
# INTREPID CAPSTONE: SILVER LAYER CLEANSING & DEDUPLICATION
# ==============================================================================

# ------------------------------------------------------------------------------
# 1. The Staging View (Decorators removed!)
# ------------------------------------------------------------------------------
@dlt.view(name="silver_orders_stg")
def silver_orders_stg():
    return (
        dlt.read_stream("bronze_orders")
        .withColumn("discount", expr("COALESCE(discount, 0.0)"))
        .withColumn("category", expr("capstone_project.capstone_schema.udf_normalize_category(category)"))
        .withColumn("customer_id", expr("capstone_project.capstone_schema.udf_mask_customer_id(customer_id)"))
        .withColumn("quality_score", expr("""
            capstone_project.capstone_schema.udf_compute_quality_score(
                customer_id, 
                product_id, 
                quantity, 
                unit_price, 
                discount
            )
        """))
        .filter("quality_score >= 0.7")
    )

# ------------------------------------------------------------------------------
# 2. The Target Table Declaration (Expectations Added Here!)
# ------------------------------------------------------------------------------
dlt.create_streaming_table(
    name="clean_orders",
    comment="Cleaned, deduplicated, and masked e-commerce orders",
    # 🛑 NEW: Add expectations directly to the materialized table! To get the health of pipeline
    expect_all_or_drop={
        "valid_order_id": "order_id IS NOT NULL",
        "valid_quantity": "quantity > 0",
        "valid_price": "unit_price >= 0.0"
    }
)

# ------------------------------------------------------------------------------
# 3. Deduplication Logic 
# ------------------------------------------------------------------------------
dlt.apply_changes(
    target="clean_orders",
    source="silver_orders_stg",
    keys=["order_id", "event_timestamp"], 
    sequence_by="processing_time", 
    ignore_null_updates=False,
    apply_as_deletes=None,
    apply_as_truncates=None,
    column_list=None
)